[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/13_Vector_Store.ipynb)

# 向量库 —— 综合指南

## 概述

本 Notebook 对 Semantica 的 vector_store 模块进行了**全面演练**,展示了向量存储、相似度搜索、混合搜索以及用于语义检索的多后端支持。

**文档**:[API 参考](https://semantica.readthedocs.io/reference/vector_store/)

### 学习目标

学完本 Notebook 后,你将能够:

- 存储和管理带元数据的向量
- 使用不同的度量方式执行相似度搜索
- 使用结合向量与元数据的混合搜索
- 使用多种向量库后端(FAISS、Weaviate 等)
- 创建和管理向量索引
- 过滤并排序搜索结果
- 为多租户实现命名空间隔离

### 你将学到什么

| 组件 | 用途 | 何时使用 |
|-----------|---------|-------------|
| `VectorStore` | 主要的向量存储 | 所有向量操作 |
| `VectorIndexer` | 索引创建 | 性能优化 |
| `VectorRetriever` | 相似度搜索 | 查找相似向量 |
| `HybridSearch` | 组合搜索 | 向量 + 元数据过滤 |
| `MetadataFilter` | 元数据过滤 | 按属性过滤 |
| `MetadataStore` | 元数据管理 | 存储向量元数据 |
| `NamespaceManager` | 多租户 | 隔离向量集合 |

---

## 安装

从 PyPI 安装 Semantica:

```bash
pip install semantica
# 或安装全部可选依赖:
pip install semantica[all]
```

---

In [ ]:
!pip install -q semantica


## 步骤 1:基础向量存储

让我们从使用 `VectorStore` 进行基础的向量存储与检索开始。

### 什么是 VectorStore?

`VectorStore` 是向量操作的主要接口:
- **存储**:存储带元数据的向量
- **搜索**:查找相似的向量
- **CRUD**:创建、读取、更新、删除操作
- **多后端**:支持 FAISS、Weaviate、Qdrant、Milvus

In [ ]:
from semantica.vector_store import VectorStore
from semantica.embeddings import TextEmbedder
import numpy as np

# 1. Initialize Embedder (Select Provider & Model)
# You can choose 'sentence_transformers' or 'fastembed'
embedder = TextEmbedder(method="sentence_transformers", model_name="all-MiniLM-L6-v2")
dimension = embedder.get_embedding_dimension()

# 2. Create vector store
store = VectorStore(backend="faiss", dimension=dimension)

# 3. Generate Real Embeddings
texts = [f"Document {i}" for i in range(100)]
vectors = embedder.embed_batch(texts)

metadata = [
    {"text": txt, "category": "science" if i % 2 == 0 else "technology", "year": 2020 + (i % 4)}
    for i, txt in enumerate(texts)
]
# 4. Store vectors
vector_ids = store.store_vectors(vectors, metadata=metadata)

print(f"Stored {len(vector_ids)} vectors")
print(f"First 3 IDs: {vector_ids[:3]}")

## 步骤 2:相似度搜索

使用不同的相似度度量搜索相似向量。

### 相似度度量

- **余弦相似度(Cosine Similarity)**:最适合语义相似度
- **L2 距离**:欧氏距离
- **点积(Dot Product)**:速度快,需要归一化的向量

In [ ]:
from semantica.vector_store import VectorIndexer, FAISSStore 
import numpy as np 
# We use the first vector from the dataset as a sample query
if 'query_vector' not in locals():
    if 'vectors' in locals() and len(vectors) > 0:
        query_vector = vectors[0]
    else:
        # Fallback if vectors are also missing (safety check)
        query_vector = np.random.rand(dimension).astype('float32')

# Create indexer 
indexer = VectorIndexer(backend="faiss", dimension=dimension) 

# Create HNSW index for fast approximate search 
adapter = FAISSStore(dimension=dimension) 
index = adapter.create_index(index_type="hnsw", metric="L2", m=16) 

# Add vectors to index 
vectors_array = np.array(vectors).astype('float32') 
# FIX: Removed 'index' argument. The adapter uses its internal self.index 
adapter.add_vectors(vectors_array, ids=vector_ids) 

# Search using index 
query_array = np.array(query_vector).astype('float32') 
# FIX: Call search on the 'index' object directly, not the adapter 
distances, indices = index.search(query_array.reshape(1, -1), k=10) 

print(f"Index search found {len(indices[0])} results") 
print(f"Distances: {distances[0][:5]}")

## 步骤 3:向量索引

为大数据集创建索引以加快搜索速度。

### 索引类型(FAISS)

- **Flat**:精确搜索(暴力搜索)
- **IVF**:倒排文件索引(近似)
- **HNSW**:层级图(兼顾最佳平衡)
- **PQ**:乘积量化(压缩)

In [ ]:
from semantica.vector_store import VectorIndexer, FAISSStore

# Get dimension from vectors to ensure consistency
dimension = len(vectors[0]) if len(vectors) > 0 else 384

# Create indexer
indexer = VectorIndexer(backend="faiss", dimension=dimension)

# Create HNSW index for fast approximate search
adapter = FAISSStore(dimension=dimension)
index = adapter.create_index(index_type="hnsw", metric="L2", m=16)

# Add vectors to index
vectors_array = np.array(vectors).astype('float32')
adapter.add_vectors(vectors_array, ids=vector_ids)

# Search using index
query_array = query_vector.astype('float32')
distances, indices = index.search(query_array.reshape(1, -1), k=10)

print(f"Index search found {len(indices[0])} results")
print(f"Distances: {distances[0][:5]}")

## 步骤 4:混合搜索

将向量相似度与元数据过滤相结合。

### 混合搜索的好处

- 在向量搜索之前先按元数据过滤
- 组合多种搜索条件
- 结果更精确

In [ ]:
from semantica.vector_store import HybridSearch, MetadataFilter 
import numpy as np

# Create hybrid search 
hybrid_search = HybridSearch() 

# Create metadata filter 
filter = MetadataFilter() \
    .eq("category", "science") \
    .gt("year", 2021) 

# Perform hybrid search 
# Ensure vectors and metadata are available
if 'vectors' not in locals() or 'metadata' not in locals() or 'vector_ids' not in locals():
    print("Warning: vectors, metadata, or vector_ids are missing. Please run previous cells.")
else:
    hybrid_results = hybrid_search.search( 
        query_vector, 
        vectors, 
        metadata, 
        vector_ids, 
        filter=filter, 
        k=10 
    ) 
    
    print(f"Hybrid search found {len(hybrid_results)} results") 
    print("\nFiltered results (science, year > 2021):") 
    for i, result in enumerate(hybrid_results[:5], 1): 
        meta = result.get('metadata', {}) 
        print(f"{i}. Category: {meta.get('category')}, Year: {meta.get('year')}, Score: {result['score']:.3f}")

## 步骤 5:元数据管理

将元数据与向量分开存储和查询。

### 元数据操作

- 为向量存储元数据
- 按元数据条件查询
- 更新元数据
- 模式(schema)校验

In [ ]:
from semantica.vector_store import MetadataStore, MetadataSchema

# Create metadata store
meta_store = MetadataStore()

# Store metadata
for i, vec_id in enumerate(vector_ids[:10]):
    meta_store.store_metadata(vec_id, metadata[i])

# Query metadata
matching_ids = meta_store.query_metadata(
    {"category": "science"},
    operator="AND"
)

print(f"Found {len(matching_ids)} vectors with category='science'")

# Define schema for validation
schema = MetadataSchema({
    "text": {"type": str, "required": True},
    "category": {"type": str, "required": True},
    "year": {"type": int, "required": True}
})

# Validate metadata
is_valid = schema.validate(metadata[0])
print(f"\nMetadata validation: {is_valid}")

## 步骤 6:结果排序与融合

合并并排序来自多次搜索的结果。

### 排序策略

- **倒数排名融合(RRF)**:合并有序列表
- **加权平均**:对来自不同来源的分数加权

In [ ]:
from semantica.vector_store import SearchRanker

# Create ranker with RRF strategy
ranker = SearchRanker(strategy="reciprocal_rank_fusion")

# Simulate multiple search results
results1 = [
    {"id": "vec_1", "score": 0.9},
    {"id": "vec_2", "score": 0.8},
    {"id": "vec_3", "score": 0.7}
]

results2 = [
    {"id": "vec_2", "score": 0.85},
    {"id": "vec_4", "score": 0.75},
    {"id": "vec_1", "score": 0.7}
]

# Fuse results using RRF
fused_results = ranker.rank([results1, results2], k=60)

print("Fused results using RRF:")
for i, result in enumerate(fused_results, 1):
    print(f"{i}. ID: {result['id']}, Fused Score: {result['score']:.3f}")

## 步骤 7:命名空间管理

为多租户应用隔离向量。

### 命名空间特性

- 租户隔离
- 访问控制
- 按命名空间的操作

In [ ]:
from semantica.vector_store import NamespaceManager

# Create namespace manager
ns_manager = NamespaceManager()

# Create namespaces for different tenants
ns1 = ns_manager.create_namespace("tenant1", "Tenant 1 vectors")
ns2 = ns_manager.create_namespace("tenant2", "Tenant 2 vectors")

# Add vectors to namespaces
for i in range(5):
    ns_manager.add_vector_to_namespace(f"t1_vec_{i}", "tenant1")
    ns_manager.add_vector_to_namespace(f"t2_vec_{i}", "tenant2")

# Get namespace vectors
tenant1_vectors = ns_manager.get_namespace_vectors("tenant1")
tenant2_vectors = ns_manager.get_namespace_vectors("tenant2")

print(f"Tenant 1: {len(tenant1_vectors)} vectors")
print(f"Tenant 2: {len(tenant2_vectors)} vectors")

# Set access control
ns1.set_access_control("user1", ["read", "write"])
ns1.set_access_control("user2", ["read"])

print(f"\nUser1 can write: {ns1.has_permission('user1', 'write')}")
print(f"User2 can write: {ns1.has_permission('user2', 'write')}")

## 步骤 8:多后端支持

使用不同的向量库后端。

### 支持的后端

| 后端 | 类型 | 最适合 |
|---------|------|----------|
| FAISS | 本地 | 开发、小数据集 |
| Weaviate | 自托管 | 感知模式的存储 |
| Qdrant | 自托管 | 高性能 |
| Milvus | 云/自托管 | 大规模 |

## 步骤 9:最佳实践

### 性能技巧

1. **归一化向量**:使用余弦相似度时务必归一化
2. **使用 HNSW**:在速度/准确度之间取得最佳平衡
3. **批量操作**:分批处理(100-1000)
4. **先过滤**:在向量搜索之前应用元数据过滤

### 后端选择

- **开发**:FAISS(本地、快速)
- **生产**:Weaviate/Qdrant(可扩展、自托管)
- **自托管**:Qdrant 或 Milvus(可控、高性能)
- **感知模式**:Weaviate(丰富的元数据)

### 索引配置

- **小数据集(<10K)**:Flat 索引
- **中等数据集(10K-1M)**:HNSW
- **大数据集(>1M)**:IVF + PQ

## 总结

### 你学到了什么

在本 Notebook 中,你学会了如何:

- 使用 VectorStore 存储和搜索向量
- 创建索引以优化性能
- 使用带元数据过滤的混合搜索
- 将元数据与向量分开管理
- 对搜索结果进行排序与融合
- 实现命名空间隔离
- 使用便捷函数进行快速操作
- 使用多种后端适配器
- 应用生产环境的最佳实践

### 关键要点

1. **多后端**:根据需求选择合适的后端
2. **混合搜索**:将向量与元数据结合以提高精度
3. **索引**:使用合适的索引类型以获得性能
4. **元数据**:单独管理元数据以获得灵活性
5. **命名空间**:为多租户隔离向量

### 下一步

**延伸阅读**:
- [Vector Store API 参考](https://semantica.readthedocs.io/reference/vector_store/)
- [高级向量库 Notebook](../advanced/Advanced_Vector_Store_and_Search.ipynb)
- [嵌入生成](12_Embedding_Generation.ipynb)

---

**有问题?**请查看我们的 [GitHub 仓库](https://github.com/semantica-agi/semantica)或[文档](https://semantica.readthedocs.io)。